In [ ]:
# 实验一：语料库处理与词向量表示
# 安装必要的库（如已安装可跳过）
%pip install -q nltk gensim jieba matplotlib numpy pandas scikit-learn


In [ ]:
import os
import re
import jieba
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import CountVectorizer

# 如果首次运行需要下载 nltk 资源可解开如下注释
# nltk.download('punkt')

DATA_CI = 'Ci.txt'
DATA_NEWS = '1998-01-2003版-带音.txt'

# 1) 读取语料，自动尝试多种编码

def load_corpus(file_path: str, encodings=None) -> str:
    if encodings is None:
        encodings = ['utf-8', 'gbk', 'gb2312', 'gb18030', 'big5']
    last_error = None
    for enc in encodings:
        try:
            with open(file_path, 'r', encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError as e:
            last_error = e
            continue
    raise ValueError(f'无法识别文件编码: {file_path}') from last_error

# 2) 文本清洗（仅保留中文、数字、字母；归一化空白）

def clean_text(text: str) -> str:
    text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 3) 分词

def segment_text(text: str):
    words = jieba.lcut(text)
    return [w for w in words if w.strip()]  # 过滤空白

ci_raw = load_corpus(DATA_CI)
news_raw = load_corpus(DATA_NEWS)

ci_clean = clean_text(ci_raw)
news_clean = clean_text(news_raw)

ci_words = segment_text(ci_clean)
news_words = segment_text(news_clean)

print('宋词样例分词:', ci_words[:20])
print('新闻样例分词:', news_words[:20])


In [ ]:
# 词频统计、可视化与 N-gram 保存
from collections import Counter


def count_word_frequency(words):
    return Counter(words)


def build_ngram(words, n=2):
    grams = []
    for i in range(len(words) - n + 1):
        grams.append(' '.join(words[i:i + n]))
    return Counter(grams)


def save_dict_to_file(counter_obj: Counter, file_path: str, top_n=None):
    pairs = counter_obj.most_common(top_n)
    with open(file_path, 'w', encoding='utf-8') as f:
        for k, v in pairs:
            f.write(f"{k}\t{v}\n")


ci_word_freq = count_word_frequency(ci_words)
news_word_freq = count_word_frequency(news_words)

# 展示 Top-20 词频
print('宋词 Top-20 词频:')
for w, c in ci_word_freq.most_common(20):
    print(w, c)
print('\n新闻 Top-20 词频:')
for w, c in news_word_freq.most_common(20):
    print(w, c)

# 可视化
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
words, freqs = zip(*ci_word_freq.most_common(20))
plt.bar(words, freqs)
plt.title('宋词 Top-20 词频')
plt.xticks(rotation=45)
plt.ylabel('频次')

plt.subplot(1, 2, 2)
words, freqs = zip(*news_word_freq.most_common(20))
plt.bar(words, freqs)
plt.title('新闻 Top-20 词频')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 生成与保存 bigram
ci_bigrams = build_ngram(ci_words, 2)
news_bigrams = build_ngram(news_words, 2)

save_dict_to_file(ci_word_freq, 'ci_word_dict.txt')
save_dict_to_file(news_word_freq, 'news_word_dict.txt')
save_dict_to_file(ci_bigrams, 'ci_bigram_dict.txt')
save_dict_to_file(news_bigrams, 'news_bigram_dict.txt')

print('已保存：ci_word_dict.txt, news_word_dict.txt, ci_bigram_dict.txt, news_bigram_dict.txt')


In [ ]:
# Word2Vec 训练、保存与可视化
from sklearn.decomposition import PCA


def prepare_sentences(words, window_size=20):
    sentences = []
    for i in range(0, len(words), window_size):
        sentences.append(words[i:i + window_size])
    return sentences


def train_word2vec(sentences, vector_size=100, window=5, min_count=5):
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4,
        sg=1  # 使用skip-gram，一般对稀疏长尾词更友好
    )
    return model


def visualize_word_vectors(model, top_n=50, title='Word Vectors Visualization'):
    words = list(model.wv.index_to_key)[:top_n]
    vectors = [model.wv[w] for w in words]
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(vectors)
    plt.figure(figsize=(12, 8))
    plt.scatter(reduced[:, 0], reduced[:, 1], c='b', alpha=0.6)
    for i, w in enumerate(words):
        plt.annotate(w, (reduced[i, 0], reduced[i, 1]))
    plt.title(title)
    plt.grid(True)
    plt.show()

# 准备句子
ci_sentences = prepare_sentences(ci_words)
news_sentences = prepare_sentences(news_words)

# 训练模型
ci_w2v = train_word2vec(ci_sentences)
news_w2v = train_word2vec(news_sentences)

# 保存模型
ci_w2v.save('ci_word2vec.model')
news_w2v.save('news_word2vec.model')

print('Word2Vec 模型已训练并保存')

# 可视化
print('宋词词向量可视化:')
visualize_word_vectors(ci_w2v, title='Ci Word Vectors (PCA)')
print('新闻词向量可视化:')
visualize_word_vectors(news_w2v, title='News Word Vectors (PCA)')


In [ ]:
# 相似度分析与 BoW 对比
from sklearn.metrics.pairwise import cosine_similarity


def analyze_word_similarity(model, query_words, topn=10):
    for w in query_words:
        if w in model.wv:
            print(f"与‘{w}’最相似的{topn}个词：")
            for sw, sim in model.wv.most_similar(w, topn=topn):
                print(f"  {sw}: {sim:.4f}")
        else:
            print(f"‘{w}’不在词表中")
        print()


def sentence_vector_average(sentence_tokens, model):
    vecs = [model.wv[w] for w in sentence_tokens if w in model.wv]
    if vecs:
        return np.mean(vecs, axis=0)
    return np.zeros(model.wv.vector_size)


def compare_bow_vs_w2v(sentences_tokens, model):
    texts = [' '.join(s) for s in sentences_tokens]
    vectorizer = CountVectorizer()
    bow = vectorizer.fit_transform(texts)
    print('BoW 维度:', bow.shape, '词表大小:', len(vectorizer.vocabulary_))
    print('W2V 维度:', model.wv.vector_size, '词表大小:', len(model.wv.index_to_key))
    if len(sentences_tokens) >= 2:
        bow_sim = cosine_similarity(bow[0:1], bow[1:2])[0][0]
        v1 = sentence_vector_average(sentences_tokens[0], model)
        v2 = sentence_vector_average(sentences_tokens[1], model)
        w2v_sim = cosine_similarity([v1], [v2])[0][0]
        print(f'前两句相似度：BoW={bow_sim:.4f}, W2V={w2v_sim:.4f}')


print('宋词相似词：')
analyze_word_similarity(ci_w2v, ['春', '月', '花', '人'])
print('新闻相似词：')
analyze_word_similarity(news_w2v, ['中国', '经济', '发展', '技术'])

print('宋词 BoW 与 W2V 对比：')
compare_bow_vs_w2v(ci_sentences[:100], ci_w2v)
print('新闻 BoW 与 W2V 对比：')
compare_bow_vs_w2v(news_sentences[:100], news_w2v)
